# JAX Frontend (`pyepo.func.jax`)

`pyepo.func.jax` mirrors `pyepo.func` so a JAX/Flax model can be trained end-to-end with `jax.grad`. Every loss keeps the same class name, constructor and call signature, and acronym alias as its PyTorch version — switching frameworks is a one-line import change:

```python
# torch:  from pyepo.func import SPOPlus
# jax:    from pyepo.func.jax import SPOPlus
```

It works with **any** PyEPO solver backend: MPAX is solved natively (GPU-native, jittable); every other backend (Gurobi, COPT, Pyomo, OR-Tools) is reached through `jax.pure_callback`.

This notebook trains a shortest-path predictor with the SPO+ loss in JAX, then swaps to the MPAX backend for a GPU-native, jitted training step. Pairs with the [JAX Frontend docs](https://khalil-research.github.io/PyEPO).

## Install (Colab only)

In [ ]:
# loss frontend + MPAX fast path, and Flax + optax for the predictor/optimizer
!pip install pyepo[mpax]
!pip install flax optax

## Optimization model and data

A 5x5 grid shortest-path model and synthetic data, exactly as in the PyTorch tutorials. The loss labels (cost, optimal solution, optimal objective) come from `optDataset`; we only convert them to JAX arrays.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp

import pyepo
from pyepo.data.dataset import optDataset

# 5x5 grid shortest path (any PyEPO solver works)
grid = (5, 5)
optmodel = pyepo.model.shortestPathModel(grid)

# synthetic data
x, c = pyepo.data.shortestpath.genData(
    num_data=1000, num_features=5, grid=grid, deg=4, noise_width=0.5, seed=135,
)
dataset = optDataset(optmodel, x, c)

# to JAX arrays
xj = jnp.asarray(x, jnp.float32)
cj = jnp.asarray(dataset.costs, jnp.float32)
wj = jnp.asarray(dataset.sols, jnp.float32)
zj = jnp.asarray(dataset.objs, jnp.float32)
print('instances:', xj.shape[0], '| cost dim:', optmodel.num_cost)

## The one-line switch

The only change from a PyTorch script is the import. The constructor kwargs (`reduction`, `processes`, `solve_ratio`, ...) and the call signature `spo(pred_cost, true_cost, true_sol, true_obj)` are identical.

In [ ]:
# PyTorch:  from pyepo.func import SPOPlus
from pyepo.func.jax import SPOPlus

spo = SPOPlus(optmodel, reduction="mean")

## End-to-end training (Flax + optax)

A linear predictor maps features to per-edge costs; `jax.grad` differentiates the SPO+ loss through the (non-differentiable) solver via its `jax.custom_vjp` gradient rule, and optax applies the update.

In [ ]:
import optax
from flax import linen as nn

# linear predictor: features -> per-edge cost
predmodel = nn.Dense(optmodel.num_cost)
params = predmodel.init(jax.random.PRNGKey(0), xj[:1])

optimizer = optax.adam(1e-2)
opt_state = optimizer.init(params)

def loss_fn(params):
    return spo(predmodel.apply(params, xj), cj, wj, zj)

for epoch in range(20):
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    if epoch % 5 == 0:
        print(f'epoch {epoch:2d}  SPO+ loss {float(loss):.4f}')

## Evaluate decision quality

Regret is the extra true cost incurred by deciding with the predicted cost instead of the true optimum, averaged over instances (here on the training set; split off a test set for real evaluation).

In [ ]:
pred_cost = np.asarray(predmodel.apply(params, xj))
true_cost = np.asarray(dataset.costs)
true_obj = np.asarray(dataset.objs).ravel()

regret = 0.0
for i in range(len(pred_cost)):
    optmodel.setObj(pred_cost[i])               # decide with the predicted cost
    sol, _ = optmodel.solve()
    regret += np.dot(true_cost[i], sol) - true_obj[i]   # true cost of that decision - optimum
print('mean regret:', regret / len(pred_cost))

## GPU with MPAX + `jax.jit`

Swap the solver backend to MPAX (no other change) and JIT the whole training step by closing over the model. MPAX batch-solves on GPU, so there is no per-step CPU round-trip. The labels (`cj`/`wj`/`zj`) are reused; only the per-step solve moves to MPAX.

In [ ]:
from pyepo.model.mpax.shortestpath import shortestPathModel as mpaxShortestPathModel

mpax_model = mpaxShortestPathModel(grid)
spo_mpax = SPOPlus(mpax_model, reduction="mean")

params = predmodel.init(jax.random.PRNGKey(0), xj[:1])
opt_state = optimizer.init(params)

@jax.jit
def step(params, opt_state):
    def loss_fn(p):
        return spo_mpax(predmodel.apply(p, xj), cj, wj, zj)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, loss

for epoch in range(20):
    params, opt_state, loss = step(params, opt_state)
    if epoch % 5 == 0:
        print(f'epoch {epoch:2d}  SPO+ loss {float(loss):.4f}')

## Notes

- **All `pyepo.func` losses are ported** — SPO+, PG, DBB/NID, DPO/DPOMul, PFY/PFYMul, I-MLE/AI-MLE, RFWO/RFY, listwise/pairwise/pointwise LTR, NCE/CMAP, CaVE — with the same class names and acronym aliases.
- **`jax.jit`**: jit the whole step by closing over the model (as above). The randomized losses (perturbed family + `implicitMLE`) are jittable when you pass an explicit `key`; `adaptiveImplicitMLE` is eager-only.
- **Caching / online pool growth** (`solve_ratio < 1`; the contrastive and ranking losses) are supported and faithful to PyTorch, but eager-only.
- **API**: every loss keeps PyTorch's signature, except `implicitMLE`/`adaptiveImplicitMLE`, which take `kappa`/`n_iterations`/`seed` scalars instead of a PyTorch `distribution` object.

See the [JAX Frontend docs](https://khalil-research.github.io/PyEPO) for the full reference.